In [16]:
"""
This script simulates a kidney exchange clearinghouse with n hospitals,
each holding a private pool of incompatible donor-patient pairs.

Two mechanisms are compared:
  1. Naive Mechanism  - no transfers; hospitals withhold "easy" pairs
  2. BP Mechanism     - Bonus-Penalty (VCG-inspired); makes truthful
                        reporting a dominant strategy

Results match Table 1 and Table 2 in the final report.
"""
import random
from collections import defaultdict
import csv
import os

def build_compatibility_graph(n_pairs, compat_prob=0.25, seed=None):
    """
    Build a random directed compatibility graph over n_pairs nodes.

    Arc (u, v) exists with probability compat_prob, meaning:
      donor of pair u is compatible with patient of pair v.

    Returns:
        adj (dict): adjacency set for each node, adj[u] = {v, ...}
    """
    if seed is not None:
        random.seed(seed)
    adj = {i: set() for i in range(n_pairs)}
    for i in range(n_pairs):
        for j in range(n_pairs):
            if i != j and random.random() < compat_prob:
                adj[i].add(j)
    return adj


def find_max_2cycles(adj, nodes):
    """
    Greedy maximum 2-cycle cover on a subgraph induced by `nodes`.

    A 2-cycle between i and j exists iff j in adj[i] AND i in adj[j].
    Each 2-cycle yields 2 transplants.

    Returns:
        n_transplants (int): total transplants from 2-cycles found
        cycles (list of tuples): list of (i, j) matched pairs
    """
    nodes = sorted(nodes)
    matched = set()
    cycles = []

    for i in nodes:
        if i in matched:
            continue
        for j in nodes:
            if j <= i or j in matched:
                continue
            # Check mutual compatibility (2-cycle)
            if j in adj.get(i, set()) and i in adj.get(j, set()):
                matched.add(i)
                matched.add(j)
                cycles.append((i, j))
                break  # move to next unmatched i

    n_transplants = 2 * len(cycles)
    return n_transplants, cycles


def generate_hospital_pools(n_hospitals, pairs_per_hospital, easy_frac):
    """
    Partition each hospital's pairs into 'easy' and 'hard' sets.
    Easy pairs: can potentially form internal 2-cycles (withheld under naive).
    Hard pairs: require cross-hospital partners (always submitted).
    Returns:
        hospitals (list of dicts): each with keys 'easy', 'hard', 'all'
        total_pairs (int): total number of pairs across all hospitals
    """
    hospitals = []
    offset = 0
    for _ in range(n_hospitals):
        n_easy = round(pairs_per_hospital * easy_frac)
        n_hard = pairs_per_hospital - n_easy
        easy_ids = list(range(offset, offset + n_easy))
        hard_ids = list(range(offset + n_easy, offset + pairs_per_hospital))
        hospitals.append({
            'easy': easy_ids,
            'hard': hard_ids,
            'all' : easy_ids + hard_ids
        })
        offset += pairs_per_hospital
    return hospitals, offset  # offset == total pairs

In [17]:
def run_single_trial(n_hospitals, pairs_per_hospital, easy_frac,
                     compat_prob, alpha, beta, audit_prob):
    """
    Simulate one exchange round under both mechanisms.
    Naive Mechanism:
      - Each hospital withholds easy pairs, matches them internally.
      - Only hard pairs submitted to clearinghouse.
      - Clearinghouse runs max-2-cycle on submitted (hard) pool.

    BP Mechanism:
      - All hospitals report truthfully (dominant strategy when
        alpha >= 1 and alpha + pi*beta >= 1).
      - Clearinghouse runs max-2-cycle on full pool.
      - Credits computed but not used to alter matching outcome
        (mechanism makes truthful reporting dominant, so no
        actual deviation occurs at equilibrium).

    Returns:
        naive_tx   (int): transplants under naive mechanism
        bp_tx      (int): transplants under BP mechanism (= social optimum)
        optimum_tx (int): social optimum transplants
        marginal   (list): Delta_i values for each hospital under BP
        credits    (list): x_i credit values for each hospital under BP
    """
    hospitals, total_pairs = generate_hospital_pools(
        n_hospitals, pairs_per_hospital, easy_frac
    )

    # Build global compatibility graph for all pairs
    adj = build_compatibility_graph(total_pairs, compat_prob)

    naive_tx = 0

    # each hospital internally matches its easy pairs
    for h in hospitals:
        internal_tx, _ = find_max_2cycles(adj, h['easy'])
        naive_tx += internal_tx

    # clearinghouse matches only hard pairs across hospitals
    hard_nodes = [p for h in hospitals for p in h['hard']]
    cross_tx, _ = find_max_2cycles(adj, hard_nodes)
    naive_tx += cross_tx

    # Social Optimum / BP Mechanism (full pool disclosed) 
    all_nodes = list(range(total_pairs))
    optimum_tx, opt_cycles = find_max_2cycles(adj, all_nodes)
    bp_tx = optimum_tx  # BP mechanism achieves optimum under truthful play

    #  Compute marginal contributions Delta_i for each hospital 
    marginal = []
    for h in hospitals:
        # Matching without hospital h's pairs
        other_nodes = [p for hh in hospitals
                         for p in hh['all'] if hh is not h]
        without_h_tx, _ = find_max_2cycles(adj, other_nodes)
        delta_i = optimum_tx - without_h_tx
        marginal.append(max(0, delta_i))

    # Compute credit transfers x_i = alpha * Delta_i - beta * withheld
    # Under truthful play withheld = 0, but we also show what a deviator faces
    credits = []
    for delta_i in marginal:
        x_i = alpha * delta_i  # no penalty under truthful reporting
        credits.append(x_i)

    return naive_tx, bp_tx, optimum_tx, marginal, credits


def run_simulation(n_hospitals=6, pairs_per_hospital=7,
                   easy_frac=0.4, compat_prob=0.25,
                   alpha=1.0, beta=1.0, audit_prob=1.0,
                   n_trials=3000, seed=42):
    """
    Run multiple trials and return summary statistics.

    Returns:
        results (dict): avg transplants, PoA, % gain, std devs
    """
    random.seed(seed)

    naive_list, bp_list, opt_list = [], [], []

    for _ in range(n_trials):
        naive_tx, bp_tx, opt_tx, _, _ = run_single_trial(
            n_hospitals, pairs_per_hospital, easy_frac,
            compat_prob, alpha, beta, audit_prob
        )
        naive_list.append(naive_tx)
        bp_list.append(bp_tx)
        opt_list.append(opt_tx)

    avg_naive = sum(naive_list) / n_trials
    avg_bp    = sum(bp_list)    / n_trials
    avg_opt   = sum(opt_list)   / n_trials

    poa  = avg_naive / avg_opt if avg_opt > 0 else 1.0
    gain = 100 * (avg_opt - avg_naive) / avg_opt if avg_opt > 0 else 0.0

    def std(lst, mean):
        return (sum((x - mean)**2 for x in lst) / len(lst)) ** 0.5

    return {
        'n_hospitals'       : n_hospitals,
        'pairs_per_hospital': pairs_per_hospital,
        'easy_frac'         : easy_frac,
        'compat_prob'       : compat_prob,
        'n_trials'          : n_trials,
        'avg_naive'         : round(avg_naive, 2),
        'avg_bp'            : round(avg_bp,    2),
        'avg_optimum'       : round(avg_opt,   2),
        'std_naive'         : round(std(naive_list, avg_naive), 2),
        'std_bp'            : round(std(bp_list,    avg_bp),    2),
        'price_of_anarchy'  : round(poa,  4),
        'pct_gain'          : round(gain, 1),
    }

In [18]:
def check_strategyproofness_conditions():
    """
    Verify which (alpha, beta) parameter combinations satisfy the
    dominant-strategy strategyproofness condition:
        alpha >= 1  AND  beta >= alpha
    (under full audit, pi = 1)

    Also verify the partial-audit condition:
        alpha + pi * beta >= 1
    """
    print("\n" + "="*62)
    print("Strategyproofness threshold (alpha, beta) pairs")
    print("Condition: alpha >= 1  AND  beta >= alpha  (pi = 1)")
    print("="*62)
    print(f"{'alpha':>8} {'beta':>8} {'Truthful dominant?':>22}")
    print("-"*42)

    for alpha in [0.5, 1.0, 1.5, 2.0]:
        for beta in [0.5, 1.0, 1.5, 2.0]:
            truthful = "YES (sufficient)" if (alpha >= 1.0 and beta >= alpha) \
                       else "no"
            print(f"{alpha:>8.1f} {beta:>8.1f} {truthful:>22}")

    print("\n" + "="*62)
    print("Partial-audit condition: alpha + pi*beta >= 1")
    print("Setting alpha = 1.0, minimum beta* = (1 - alpha) / pi")
    print("="*62)
    print(f"{'pi (audit prob)':>18} {'beta* (min required)':>22}")
    print("-"*42)
    alpha = 1.0
    for pi in [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
        beta_star = max(0.0, (1 - alpha) / pi) if pi > 0 else float('inf')
        note = " <- bonus alone sufficient" if beta_star == 0.0 else ""
        print(f"{pi:>18.1f} {beta_star:>22.2f}{note}")



def experiment_vary_easy_frac(n_trials=3000):
    """
    Table 1 in the report: vary easy_frac from 0.1 to 0.7.
    Fixed: n_hospitals=6, pairs_per_hospital=7, compat_prob=0.25
    """
    
    print("\n" + "="*75)
    print("Transplant counts by easy-pair fraction phi")
    print("(6 hospitals, 7 pairs each, compat_prob=0.25, "
          f"n_trials={n_trials})")
    print("="*75)
    print(f"{'phi':>6} | {'Naive':>8} | {'BP Mech':>9} | "
          f"{'Optimum':>9} | {'PoA':>7} | {'% Gain':>8}")
    print("-"*60)

    rows = []
    for ef in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
        r = run_simulation(easy_frac=ef, n_trials=n_trials)
        print(f"{ef:>6.1f} | {r['avg_naive']:>8.2f} | "
              f"{r['avg_bp']:>9.2f} | {r['avg_optimum']:>9.2f} | "
              f"{r['price_of_anarchy']:>7.3f} | {r['pct_gain']:>7.1f}%")
        rows.append(r)
    return rows


def experiment_vary_hospitals(n_trials=2000):
    """
    Additional experiment: vary number of hospitals (pool size effect).
    """
    print("\n" + "="*75)
    print("Effect of number of hospitals (easy_frac=0.4)")
    print("="*75)
    print(f"{'n_h':>5} | {'Naive':>8} | {'BP Mech':>9} | "
          f"{'Optimum':>9} | {'PoA':>7} | {'% Gain':>8}")
    print("-"*60)

    rows = []
    for n in [3, 5, 8, 10, 15, 20]:
        r = run_simulation(n_hospitals=n, easy_frac=0.4, n_trials=n_trials)
        print(f"{n:>5} | {r['avg_naive']:>8.2f} | "
              f"{r['avg_bp']:>9.2f} | {r['avg_optimum']:>9.2f} | "
              f"{r['price_of_anarchy']:>7.3f} | {r['pct_gain']:>7.1f}%")
        rows.append(r)
    return rows


def experiment_single_example():
    """
    Illustrative single-trial walkthrough showing the mechanism in detail.
    """
    print("\n" + "="*62)
    print("EXAMPLE: Single trial walkthrough (n=3 hospitals, 4 pairs each)")
    print("="*62)

    random.seed(7)
    n_h, pph, ef = 3, 4, 0.5
    hospitals, total = generate_hospital_pools(n_h, pph, ef)
    adj = build_compatibility_graph(total, compat_prob=0.3, seed=7)

    print(f"\nHospital pools (easy | hard):")
    for i, h in enumerate(hospitals):
        print(f"  Hospital {i+1}: easy={h['easy']}  hard={h['hard']}")

    print("\nCompatibility arcs (u -> v means donor_u compatible with patient_v):")
    for u in range(total):
        if adj[u]:
            print(f"  {u} -> {sorted(adj[u])}")

    # Naive mechanism
    naive_tx = 0
    print("\n-- Naive Mechanism --")
    for i, h in enumerate(hospitals):
        tx, cyc = find_max_2cycles(adj, h['easy'])
        print(f"  Hospital {i+1} internal easy matches: {cyc}  (+{tx} tx)")
        naive_tx += tx
    hard_nodes = [p for h in hospitals for p in h['hard']]
    cross_tx, cross_cyc = find_max_2cycles(adj, hard_nodes)
    print(f"  Cross-hospital hard matches: {cross_cyc}  (+{cross_tx} tx)")
    naive_tx += cross_tx
    print(f"  TOTAL (naive): {naive_tx} transplants")

    # BP mechanism (full pool)
    all_nodes = list(range(total))
    opt_tx, opt_cyc = find_max_2cycles(adj, all_nodes)
    print("\n-- BP Mechanism (full pool) --")
    print(f"  Global matches: {opt_cyc}  (+{opt_tx} tx)")
    print(f"  TOTAL (BP / optimum): {opt_tx} transplants")
    print(f"  Gain: {opt_tx - naive_tx} transplants "
          f"({100*(opt_tx-naive_tx)/opt_tx:.0f}% improvement)"
          if opt_tx > 0 else "")

    # Marginal contributions and credits
    print("\n-- Credit transfers (alpha=1.0) --")
    for i, h in enumerate(hospitals):
        other = [p for hh in hospitals for p in hh['all'] if hh is not h]
        without_tx, _ = find_max_2cycles(adj, other)
        delta = max(0, opt_tx - without_tx)
        credit = 1.0 * delta
        print(f"  Hospital {i+1}: Delta={delta}  credit x_i={credit:.1f}")


def save_results_csv(rows, filename):
    if not rows:
        return
        
    # os.getcwd() works perfectly in Jupyter Notebooks
    script_dir = os.getcwd() 
    path = os.path.join(script_dir, filename)
    
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f"\n  Results saved to: {path}")

In [19]:
# Illustrative walkthrough first
experiment_single_example()


EXAMPLE: Single trial walkthrough (n=3 hospitals, 4 pairs each)

Hospital pools (easy | hard):
  Hospital 1: easy=[0, 1]  hard=[2, 3]
  Hospital 2: easy=[4, 5]  hard=[6, 7]
  Hospital 3: easy=[8, 9]  hard=[10, 11]

Compatibility arcs (u -> v means donor_u compatible with patient_v):
  0 -> [2, 4, 7, 9, 11]
  1 -> [0, 4, 5, 11]
  2 -> [1, 3, 4, 7]
  3 -> [0, 1, 2, 9]
  4 -> [0, 6, 8, 11]
  5 -> [1]
  6 -> [4, 10]
  7 -> [1, 3, 4, 5, 8, 9]
  8 -> [0, 6]
  9 -> [0, 1, 2, 3, 6, 7]
  10 -> [5]
  11 -> [1, 3, 4, 5, 6, 8, 9, 10]

-- Naive Mechanism --
  Hospital 1 internal easy matches: []  (+0 tx)
  Hospital 2 internal easy matches: []  (+0 tx)
  Hospital 3 internal easy matches: []  (+0 tx)
  Cross-hospital hard matches: [(2, 3)]  (+2 tx)
  TOTAL (naive): 2 transplants

-- BP Mechanism (full pool) --
  Global matches: [(0, 4), (1, 5), (2, 3), (7, 9)]  (+8 tx)
  TOTAL (BP / optimum): 8 transplants
  Gain: 6 transplants (75% improvement)

-- Credit transfers (alpha=1.0) --
  Hospital 1: Delt

In [20]:
rows1 = experiment_vary_easy_frac(n_trials=3000)
save_results_csv(rows1, "sim_table1_easy_frac.csv")


Transplant counts by easy-pair fraction phi
(6 hospitals, 7 pairs each, compat_prob=0.25, n_trials=3000)
   phi |    Naive |   BP Mech |   Optimum |     PoA |   % Gain
------------------------------------------------------------
   0.1 |    25.81 |     31.48 |     31.48 |   0.820 |    18.0%
   0.2 |    25.81 |     31.48 |     31.48 |   0.820 |    18.0%
   0.3 |    20.85 |     31.48 |     31.48 |   0.662 |    33.8%
   0.4 |    16.83 |     31.48 |     31.48 |   0.534 |    46.6%
   0.5 |    13.65 |     31.48 |     31.48 |   0.434 |    56.6%
   0.6 |    13.65 |     31.48 |     31.48 |   0.434 |    56.6%
   0.7 |    11.36 |     31.48 |     31.48 |   0.361 |    63.9%

  Results saved to: /home/shreyas/Desktop/igt/end_use/sim_table1_easy_frac.csv


In [21]:
check_strategyproofness_conditions()


Strategyproofness threshold (alpha, beta) pairs
Condition: alpha >= 1  AND  beta >= alpha  (pi = 1)
   alpha     beta     Truthful dominant?
------------------------------------------
     0.5      0.5                     no
     0.5      1.0                     no
     0.5      1.5                     no
     0.5      2.0                     no
     1.0      0.5                     no
     1.0      1.0       YES (sufficient)
     1.0      1.5       YES (sufficient)
     1.0      2.0       YES (sufficient)
     1.5      0.5                     no
     1.5      1.0                     no
     1.5      1.5       YES (sufficient)
     1.5      2.0       YES (sufficient)
     2.0      0.5                     no
     2.0      1.0                     no
     2.0      1.5                     no
     2.0      2.0       YES (sufficient)

Partial-audit condition: alpha + pi*beta >= 1
Setting alpha = 1.0, minimum beta* = (1 - alpha) / pi
   pi (audit prob)   beta* (min required)
----------------

In [22]:
rows2 = experiment_vary_hospitals(n_trials=2000)
save_results_csv(rows2, "sim_extra_hospitals.csv")


Effect of number of hospitals (easy_frac=0.4)
  n_h |    Naive |   BP Mech |   Optimum |     PoA |   % Gain
------------------------------------------------------------
    3 |     6.13 |     12.09 |     12.09 |   0.507 |    49.3%
    5 |    13.11 |     24.89 |     24.89 |   0.527 |    47.3%
    8 |    24.77 |     45.19 |     45.19 |   0.548 |    45.2%
   10 |    33.15 |     59.10 |     59.10 |   0.561 |    43.9%
   15 |    54.34 |     94.00 |     94.00 |   0.578 |    42.2%
   20 |    76.00 |    128.98 |    128.98 |   0.589 |    41.1%

  Results saved to: /home/shreyas/Desktop/igt/end_use/sim_extra_hospitals.csv
